In [ ]:
!pip install peft bitsandbytes datasets safetensors accelerate -q

import os, json, torch, shutil, gc
import bitsandbytes as bnb
import safetensors.torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from torch.optim import AdamW
from datasets import load_dataset, Dataset
import random
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

MODEL_NAME   = "QwenRouterAI"
MODEL_AUTHOR = "China Unicom"
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"

print("Loading dataset...")
raw_ds = load_dataset("modelscope/self-cognition", split="train")
def fill_placeholders(example):
    return {
        "query": example["query"].replace("{{NAME}}", MODEL_NAME).replace("{{AUTHOR}}", MODEL_AUTHOR),
        "response": example["response"].replace("{{NAME}}", MODEL_NAME).replace("{{AUTHOR}}", MODEL_AUTHOR)
    }

templates = [fill_placeholders(raw_ds[i]) for i in range(len(raw_ds))]

rows = [random.choice(templates) for _ in range(1000)]
ds = Dataset.from_list(rows)

print(f"Dataset size: {len(ds)}")
print(f"Q: {ds[0]['query']}\nA: {ds[0]['response'][:80]}...")

print("Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")

peft_config = LoraConfig(r=16, lora_alpha=32, task_type=TaskType.CAUSAL_LM, target_modules=["q_proj","v_proj"], inference_mode=False)
model = get_peft_model(model, peft_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

def tokenize(example):
    prompt = f"<|im_start|>user\n{example['query']}<|im_end|>\n<|im_start|>assistant\n{example['response']}<|im_end|>"
    tokens = tokenizer(prompt, truncation=True, max_length=512, padding="max_length", return_tensors="pt")
    tokens["labels"] = tokens["input_ids"].clone()
    return {k: v.squeeze(0) for k, v in tokens.items()}

tokenized = ds.map(tokenize, remove_columns=ds.column_names)

from torch.utils.data import DataLoader
def collate_fn(batch):
    return {k: torch.stack([torch.tensor(item[k]) for item in batch]) for k in batch[0]}
loader = DataLoader(tokenized, batch_size=1, shuffle=True, collate_fn=collate_fn)
optimizer = AdamW(model.parameters(), lr=1e-4)

print("Training...")
model.train()
for epoch in range(5):
    epoch_loss, epoch_steps = 0.0, 0
    for step, batch in enumerate(loader):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        torch.cuda.empty_cache()
        epoch_loss += loss.item()
        epoch_steps += 1
        if (step+1) % 10 == 0:
            print(f"  Epoch {epoch+1} step {step+1}/{len(loader)} loss={loss.item():.4f} avg={epoch_loss/epoch_steps:.4f}")
    print(f"Epoch {epoch+1} done. Avg loss: {epoch_loss/epoch_steps:.4f}")

# Save config and tokenizer while model is still in memory
os.makedirs("/kaggle/working/qwen-fp16", exist_ok=True)
model.config.save_pretrained("/kaggle/working/qwen-fp16")
tokenizer.save_pretrained("/kaggle/working/qwen-fp16")

print("Dequantizing and saving in chunks...")
CHUNK_SIZE = 50
shard_idx = 1
current_chunk = {}
weight_map = {}

all_items = []
for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        all_items.append((name + ".weight", "4bit", module))
        if module.bias is not None:
            all_items.append((name + ".bias", "bias", module))
    elif isinstance(module, torch.nn.Linear):
        all_items.append((name + ".weight", "linear", module))
        if module.bias is not None:
            all_items.append((name + ".bias", "bias_linear", module))

for name, param in model.named_parameters():
    if not any(name == n for n, _, _ in all_items):
        all_items.append((name, "param", param))

print(f"Total tensors: {len(all_items)}")

for name, kind, obj in all_items:
    if kind == "4bit":
        t = bnb.functional.dequantize_4bit(obj.weight.data, obj.weight.quant_state).to(torch.float16).cpu()
    elif kind == "bias":
        t = obj.bias.data.to(torch.float16).cpu()
    elif kind == "linear":
        t = obj.weight.data.to(torch.float16).cpu()
    elif kind == "bias_linear":
        t = obj.bias.data.to(torch.float16).cpu()
    else:
        t = obj.data.to(torch.float16).cpu()

    current_chunk[name] = t
    if len(current_chunk) >= CHUNK_SIZE:
        shard_name = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
        safetensors.torch.save_file(current_chunk, f"/kaggle/working/qwen-fp16/{shard_name}")
        for k in current_chunk:
            weight_map[k] = shard_name
        print(f"  Saved shard {shard_idx}")
        shard_idx += 1
        current_chunk = {}
        gc.collect()

if current_chunk:
    shard_name = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
    safetensors.torch.save_file(current_chunk, f"/kaggle/working/qwen-fp16/{shard_name}")
    for k in current_chunk:
        weight_map[k] = shard_name
    shard_idx += 1

total_shards = shard_idx - 1
del model
torch.cuda.empty_cache()
shutil.rmtree(os.path.expanduser("~/.cache/huggingface"), ignore_errors=True)
gc.collect()

# Rename shards and fix tensor names (strip PEFT prefix) in one pass
print("Fixing tensor names...")
final_weight_map = {}
for i in range(1, total_shards + 1):
    old_path = f"/kaggle/working/qwen-fp16/model-{i:05d}-of-XXXXX.safetensors"
    new_shard = f"model-{i:05d}-of-{total_shards:05d}.safetensors"
    new_path = f"/kaggle/working/qwen-fp16/{new_shard}"
    
    from safetensors import safe_open
    chunk = {}
    with safe_open(old_path, framework="pt", device="cpu") as f:
        for key in f.keys():
            new_key = key
            if new_key.startswith("base_model.model.model."):
                new_key = "model." + new_key[len("base_model.model.model."):]
            elif new_key.startswith("base_model.model."):
                new_key = new_key[len("base_model.model."):]
            if ".lora_" in new_key:
                continue
            if ".base_layer." in new_key:
                new_key = new_key.replace(".base_layer.", ".")
            chunk[new_key] = f.get_tensor(key)
    
    safetensors.torch.save_file(chunk, new_path)
    os.remove(old_path)
    for k in chunk:
        final_weight_map[k] = new_shard
    print(f"  Fixed shard {i}: {len(chunk)} tensors")
    del chunk; gc.collect()

# Fix config and tokenizer config
with open("/kaggle/working/qwen-fp16/config.json") as f:
    cfg = json.load(f)
cfg.pop("quantization_config", None)
with open("/kaggle/working/qwen-fp16/config.json", "w") as f:
    json.dump(cfg, f, indent=2)

with open("/kaggle/working/qwen-fp16/tokenizer_config.json") as f:
    tc = json.load(f)
if isinstance(tc.get("extra_special_tokens"), list):
    tc["extra_special_tokens"] = {}
with open("/kaggle/working/qwen-fp16/tokenizer_config.json", "w") as f:
    json.dump(tc, f, indent=2)

index = {"metadata": {"total_size": 0}, "weight_map": final_weight_map}
with open("/kaggle/working/qwen-fp16/model.safetensors.index.json", "w") as f:
    json.dump(index, f, indent=2)

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free disk: {free/1024**3:.1f} GB")